In [1]:
secfiles = [
    "sections-data-tool2-shumei.json",
    "sections-shumei-seqingyouxi.json",
    "sections-data-tool-shumei.json",
    "sections-nsfw.json",
    "sections-nsfw-shumei.json",
    "sections-safety240722.json",
]

import json

secdata = {}
seccounts = {}

for sf in secfiles:
    data = json.load(open(sf))
    secdata[sf] = data

    counts = seccounts[sf] = {}

    for k, v in data["count"].items():
        counts[k] = v

from pprint import pprint

pprint(seccounts)

{'sections-data-tool-shumei.json': {'ban': 4886,
                                    'blackandwhitelist': 21,
                                    'minor': 195,
                                    'normal': 118249,
                                    'politics-1001': 7374,
                                    'politics-1002': 11961,
                                    'porn-1001': 1003,
                                    'porn-1002': 3023,
                                    'sexy': 0,
                                    'star': 347,
                                    'violence': 923},
 'sections-data-tool2-shumei.json': {'normal': 181292,
                                     'politics-1001': 23866,
                                     'politics-1002': 46439,
                                     'porn-1001': 485,
                                     'porn-1002': 401},
 'sections-nsfw-shumei.json': {'normal': 3111, 'porn': 4408},
 'sections-nsfw.json': {'drawings': 9646,
               

In [2]:
import random
from copy import deepcopy

collects = {
    # 0930
    "sections-data-tool2-shumei.json": {
        "normal": {"train": 0.32, "val": 350, "label": "normal"},
        "politics-1002": {"train": 2.0, "val": 500, "label": "politics"},
        "porn-1002": {"train": 1.0, "val": 50, "label": "porn"},
    },
    "sections-shumei-seqingyouxi.json": {
        "normal": {"train": 1.0, "val": 50, "label": "normal"},
        "porn": {"train": 5.0, "val": 50, "label": "porn"},
    },
    # 以前
    "sections-data-tool-shumei.json": {
        "normal": {"train": 0.32, "val": 300, "label": "normal"},
        "politics-1002": {"train": 2.0, "val": 500, "label": "politics"},
        "porn-1002": {"train": 1.0, "val": 350, "label": "porn"},
    },
    "sections-nsfw-shumei.json": {
        "normal": {"train": 1.0, "val": 100, "label": "normal"},
        "porn": {"train": 1.0, "val": 350, "label": "porn"},
    },
    "sections-nsfw.json": {
        "drawings": {"train": 0.5, "val": 100, "label": "normal"},
        "neutral": {"train": 0.5, "val": 100, "label": "normal"},
    },
    "sections-safety240722.json": {
        "20240322_8k_nsfw_dup_1507": {"train": 0.8, "val": 50, "label": "porn"},
        "20240329_163k_nsfw_dup_110k": {"train": 0.8, "val": 50, "label": "porn"},
        "nsfw_23k": {"train": 0.8, "val": 50, "label": "porn"},
        "sq_8k": {"train": 0.8, "val": 50, "label": "porn"},
    },
}

train_counts = {}
val_counts = {}
train_anns = []
val_anns = []
labels = []
label_map = {}

random.seed(0)

for sf, col in collects.items():
    # for each file, collect multiple sections
    counts = seccounts[sf]
    data = secdata[sf]

    for k, v in col.items():
        # collect sections

        label = v["label"]
        val_count = int(v.get("val",0))
        train_count = int((counts[k] - val_count) * v["train"])

        # stat
        if label in train_counts:
            train_counts[label] += train_count
        else:
            train_counts[label] = train_count
        if label in val_counts:
            val_counts[label] += val_count
        else:
            val_counts[label] = val_count
        if label not in labels:
            labels.append(label)
            label_map[label] = len(labels) - 1
        label_id = label_map[label]

        # collect data
        files = deepcopy(data["files"][k])
        random.shuffle(files)

        # get val first to avoid overlap
        for _ in range(val_count):
            val_anns.append((files.pop(), label_id))

        # sample train
        files = files * int(v["train"] + 1)
        files = files[: int(train_count)]

        train_ann_ = [(f, label_id) for f in files]
        train_anns.extend(train_ann_)

pprint(train_counts)
pprint(val_counts)
pprint(train_anns[::len(train_anns)//20])
pprint(val_anns[::len(val_anns)//20])

assert set(train_anns) & set(val_anns) == set()

{'normal': 115820, 'politics': 114800, 'porn': 111426}
{'normal': 1000, 'politics': 1000, 'porn': 1000}
[('data_tool2/res2/陈光诚/google_陈光诚/00348.jpeg', 0),
 ('data_tool2/res2/蔡英文/google_蔡英文/00353.jpeg', 0),
 ('data_tool2/res2/普京/google_普京/00457.jpeg', 0),
 ('data_tool2/res/virus king/google_virus king/00014.jpeg', 0),
 ('data_tool2/res2/丧事喜办/google_丧事喜办/00550.jpeg', 1),
 ('data_tool2/res2/多名女性/google_多名女性/00151.jpeg', 1),
 ('data_tool2/res2/宋志标/google_宋志标/00061.jpeg', 1),
 ('data_tool2/res2/派系/google_派系/00457.jpeg', 1),
 ('data_tool2/res2/乌尔凯西/google_乌尔凯西/00049.jpeg', 1),
 ('pachong2/huang/seqingyouxiguanggao/1300033213.png', 2),
 ('data_tool/res/成人用品/google_情趣用品/00088.jpeg', 0),
 ('data_tool/res/特殊语种/google_韩语/00240.jpeg', 0),
 ('data_tool/res/政治映射/google_会晤场景/00169.jpeg', 1),
 ('raw_data/hentai/IMAGES/tcy5g2wko0o31.jpg', 2),
 ('raw_data/neutral/IMAGES/8lhplxnu3li21.jpg', 0),
 ('safety240722/seqing/20240329_163k_nsfw_dup_110k/111513.jpg', 2),
 ('safety240722/seqing/20240329_163k_nsfw_d

In [3]:
with open('huangfan-shumei-1021-train.txt', 'w') as f:
    for ann in train_anns:
        f.write(f"{ann[0]} {ann[1]}\n")

with open('huangfan-shumei-1021-val.txt', 'w') as f:
    for ann in val_anns:
        f.write(f"{ann[0]} {ann[1]}\n")

with open('huangfan-shumei-1021-labels.txt', 'w') as f:
    for label in labels:
        f.write(f"{label}\n")